# Perturbatively Flat Vacua and Conifold PFVs

**What's in this notebook?** This notebook demonstrates the perturbatively flat vacuum (PFV) mechanism for constructing flux vacua with exponentially small $|W_0|$. We cover:

1. **Standard PFVs** at large complex structure — the polynomial superpotential vanishes exactly along a flat direction, leaving only exponentially suppressed instanton corrections
2. **Conifold PFVs** — the PFV mechanism extended to geometries with conifold singularities, engineering warped Klebanov-Strassler throats

**In this notebook, you will learn:**
- How to construct PFV flux vectors using `pfv_to_flux` and verify the PFV conditions
- How to compute the PFV flat direction via `pfv_to_moduli`
- How to refine the PFV initial guess to a true SUSY vacuum using Newton's method
- How the conifold PFV extends this to the `coniLCS` limit with the `ConifoldFreezer`

**Prerequisites:** [Tutorial 5: Finding flux vacua](../02_vacuum_finding/5_finding_flux_vacua.ipynb), [Tutorial 13: coniLCS](../04_geometry_and_limits/13_coniLCS.ipynb).

**References:**
- PFV mechanism: [1903.00596](https://arxiv.org/abs/1903.00596)
- Conifold PFVs: [2004.10740](https://arxiv.org/abs/2004.10740)
- TASI lectures (Sections 6.2, 6.4): [2512.17095](https://arxiv.org/abs/2512.17095)
- See also the [PFV introduction](../intro/pfv) for the theoretical background.

(*Created:* March 2026)

## Imports

In [ ]:
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import jax
import jax.numpy as jnp
jax.config.update("jax_enable_x64", True)
from jax import vmap

import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams.update({"figure.dpi": 200, "font.size": 10, "axes.labelsize": 11,
                      "figure.figsize": (3.4, 2.8), "legend.frameon": False,
                      "xtick.direction": "in", "ytick.direction": "in"})

import jaxvacua as jvc
from jaxvacua.freezer import Freezer, ConifoldFreezer

## Part 1: Standard PFVs at Large Complex Structure

### The PFV mechanism

The flux superpotential splits as $W_{\mathrm{flux}} = W_{\mathrm{poly}} + W_{\mathrm{exp}}$. 
By choosing integer fluxes $(M^a, K_a)$ satisfying the **PFV conditions**:

$$
\det N \neq 0\,, \quad \vec{p} \in \mathcal{K}_{\widetilde{X}}\,, \quad K_a p^a = 0\,, \quad \tilde{a}_{ab} M^b \in \mathbb{Z}\,, \quad \tilde{c}_a M^a \in 24\mathbb{Z}\,,
$$

with $N_{ab} = \kappa_{abc} M^c$ and $p^a = N^{ab} K_b$, the polynomial part vanishes exactly along the one-dimensional locus $z^a = p^a \tau$.

The remaining superpotential is purely exponential:

$$
|W_0| \sim |W_{\mathrm{exp}}| \ll 1\,.
$$

### Setup: degree-18 hypersurface

We use the degree-18 hypersurface in $\mathbb{CP}_{[1,1,1,6,9]}$ with $h^{2,1} = 2$ (on the $\mathbb{Z}_6 \times \mathbb{Z}_{18}$-invariant locus) and $Q_{\mathrm{D3}} = 138$. This is the canonical PFV example from [1903.00596](https://arxiv.org/abs/1903.00596) and [2512.17095](https://arxiv.org/abs/2512.17095), Section 6.2.

In [ ]:
model = jvc.FluxVacuaFinder(h12=2, model_ID=1, maximum_degree=2, limit="LCS", model_type="KS")
print(model)
print(f"D3 tadpole: {model.D3_tadpole}")
print(f"Intersection numbers: {model.lcs_tree.intnums.astype(int)}")

### Constructing PFV fluxes

The PFV flux ansatz is:

$$
\vec{f} = (R_0, R_a, 0, M^a)^\top\,, \quad \vec{h} = (0, K_a, 0, 0^a)^\top\,,
$$

where $R_0$ and $R_a$ are fixed by the integrality conditions. The `pfv_to_flux` method constructs the full flux vector from $(M^a, K_a)$.

In [ ]:
# PFV flux vectors from Section 6.2 of arXiv:2512.17095
M = jnp.array([-16., 50.])
K = jnp.array([3., -4.])

# Construct the full flux vector
flux_pfv = model.pfv_to_flux(M, K)
print(f"Full flux vector: {flux_pfv.astype(int)}")

# Verify tadpole
Nflux = model.tadpole(flux_pfv)
print(f"N_flux = {int(Nflux)}  (must be ≤ {model.D3_tadpole})")

### Verifying the PFV conditions

In [ ]:
# N_{ab} = kappa_{abc} M^c
kappa = model.lcs_tree.intnums
N = jnp.einsum('ijk,k->ij', kappa, M)
print(f"N_ab = \n{N.astype(int)}")
print(f"det(N) = {int(jnp.linalg.det(N))}  (must be ≠ 0) ✓")

# p^a = N^{-1} K
p = jnp.linalg.inv(N) @ K
print(f"\np = {p}  (direction vector)")
print(f"K · p = {float(K @ p):.6e}  (must be 0) ✓")

# Integrality: a_matrix @ M must be integer
a = model.lcs_tree.a_matrix
aM = a @ M
print(f"\na_ab M^b = {aM}  (must be integer) ✓")

# c-vector check: c · M must be divisible by 24
b = model.lcs_tree.b_vector
cM = 24 * b @ M
print(f"c_a M^a = {float(cM):.0f}  (must be in 24Z) ✓")

### Computing the PFV flat direction

Along the PFV locus $z^a = p^a \tau$, the polynomial superpotential vanishes. The `pfv_to_moduli` method computes the moduli at this locus for a given $\tau$.

In [ ]:
# Evaluate |W| along the flat direction for a range of Im(tau)
tau_values = jnp.linspace(4, 12, 200) * 1j
W_along_flat = []

for tau_v in tau_values:
    z_v = model.pfv_to_moduli(M, K, tau_v)
    W_v = model.superpotential(z_v, tau_v, flux_pfv)
    W_along_flat.append(float(jnp.abs(W_v)))

W_along_flat = np.array(W_along_flat)

fig, ax = plt.subplots()
ax.semilogy(np.array(tau_values.imag), W_along_flat, color="#2563eb", lw=1.5)
ax.set_xlabel(r"Im$(\tau)$")
ax.set_ylabel(r"$|W_{\mathrm{flux}}|$")
ax.set_title("Superpotential along the PFV flat direction")
ax.axvline(x=6.86, color="#dc2626", ls="--", lw=0.8, alpha=0.7, label=r"$\langle\tau\rangle$")
ax.legend()
fig.tight_layout()
plt.show()

# The minimum gives the racetrack vacuum
idx_min = np.argmin(W_along_flat)
print(f"Minimum |W| = {W_along_flat[idx_min]:.4e} at Im(tau) = {float(tau_values[idx_min].imag):.2f}")

### Newton refinement to the exact vacuum

The PFV flat direction provides an excellent initial guess. We refine with Newton's method (real solver mode, damped step size) to find the exact SUSY vacuum where $D_aW = 0$.

In [ ]:
# Initial guess from PFV
tau_init = 6.85j
z_init = model.pfv_to_moduli(M, K, tau_init)

print(f"PFV initial guess:")
print(f"  z   = {z_init}")
print(f"  tau = {tau_init}")
print(f"  |W| = {float(jnp.abs(model.superpotential(z_init, tau_init, flux_pfv))):.4e}")

# Newton refinement (real solver, damped)
z_sol, tau_sol, res = model.newton_method_flux_vacua(
    z_init, tau_init, flux_pfv,
    mode=None, step_size_Newton=0.1,
    tol=1e-12, max_iters=500, solver_mode="real"
)

# Results
DW = model.DW(z_sol, jnp.conj(z_sol), tau_sol, jnp.conj(tau_sol), flux_pfv)
W0 = model.superpotential(z_sol, tau_sol, flux_pfv)
W0_gi = model.superpotential_gauge_invariant(z_sol, tau_sol, flux_pfv)

print(f"\nPFV vacuum found:")
print(f"  z   = {z_sol}")
print(f"  tau = {tau_sol}")
print(f"  |DW|  = {float(jnp.sum(jnp.abs(DW))):.2e}")
print(f"  |W|   = {float(jnp.abs(W0)):.6e}")
print(f"  |W0| (gauge-invariant) = {float(jnp.abs(W0_gi)):.6e}")
print(f"  N_flux = {int(model.tadpole(flux_pfv))}")
print(f"  Residual = {float(res):.2e}")

### The racetrack mechanism

The exponentially small $|W_0|$ arises from a **racetrack** — two competing instanton contributions with nearly equal exponents. The effective superpotential along the flat direction is:

$$
W_{\mathrm{eff}}(\tau) = c \left( e^{2\pi i p^1 \tau} + A\, e^{2\pi i p^2 \tau} \right) + \ldots
$$

When $|p^1 - p^2| \ll p^2$, the two terms nearly cancel at the minimum, producing $|W_0| \ll 1$.

In [ ]:
# Show the two leading instanton contributions
print(f"PFV direction: p = {p}")
print(f"p1 = {float(p[0]):.4f}")
print(f"p2 = {float(p[1]):.4f}")
print(f"|p1 - p2| = {float(jnp.abs(p[0] - p[1])):.4f}")
print(f"|p1 - p2| / p2 = {float(jnp.abs(p[0] - p[1]) / jnp.abs(p[1])):.4f}")
print()
print(f"At the vacuum:")
print(f"  exp(2πi p1 τ) ~ exp(-2π × {float(p[0]):.3f} × {float(tau_sol.imag):.3f})")
print(f"                = {float(jnp.exp(-2*jnp.pi*p[0]*tau_sol.imag)):.4e}")
print(f"  exp(2πi p2 τ) ~ exp(-2π × {float(p[1]):.3f} × {float(tau_sol.imag):.3f})")
print(f"                = {float(jnp.exp(-2*jnp.pi*p[1]*tau_sol.imag)):.4e}")
digits = int(-np.log10(float(jnp.exp(-2*jnp.pi*p[0]*tau_sol.imag))))
print(f"\n→ Both contributions are O(10^-{digits}) and nearly cancel → |W0| ≈ {float(jnp.abs(W0_gi)):.2e}")

## Part 2: Conifold PFVs

### Extending PFVs to conifold geometries

Near a conifold singularity, the prepotential acquires a logarithmic correction:

$$
\mathcal{F}(z_{\mathrm{cf}}, z^\alpha) = n_{\mathrm{cf}} \frac{z_{\mathrm{cf}}^2}{4\pi i} \ln(-2\pi i\, z_{\mathrm{cf}}) + \sum_n \frac{\mathcal{F}^{(n)}(z^\alpha)}{n!} z_{\mathrm{cf}}^n
$$

The conifold modulus is stabilised at an exponentially small value:

$$
\langle |z_{\mathrm{cf}}| \rangle = \frac{1}{2\pi} \exp\!\left(-\frac{2\pi K'}{g_s M\, n_{\mathrm{cf}}}\right)
$$

giving rise to a warped Klebanov-Strassler throat.

### Loading a conifold model

In [ ]:
# Load a coniLCS model (h12=3, from arXiv:2009.03312)
try:
    model_coni = jvc.FluxVacuaFinder(h12=3, model_ID=1, limit="coniLCS", model_type="KS")
    print(model_coni)
    print(f"h12 = {model_coni.h12}")
    print(f"limit = {model_coni.periods.limit}")
    print(f"n_cf = {model_coni.lcs_tree.ncf}")
    HAS_CONI = True
except Exception as e:
    print(f"ConiLCS model not available: {e}")
    print("\nSkipping conifold PFV section (requires pre-computed coniLCS model data).")
    HAS_CONI = False

### Conifold PFV construction

For a conifold PFV, the bulk moduli $z^\alpha$ are stabilised via the same PFV mechanism as before, while the conifold modulus $z_{\mathrm{cf}}$ is stabilised by the logarithmic monodromy.

The **conifold PFV conditions** are the same as the standard PFV conditions but applied to the *bulk* indices $\alpha$ only:

$$
\det N \neq 0\,, \quad \vec{p} \in \mathcal{K}_{\mathrm{cf}}\,, \quad K_\alpha p^\alpha = 0\,, \quad \tilde{a}_{\alpha b} M^b \in \mathbb{Z}\,, \quad \tilde{c}'_a M^a \in 24\mathbb{Z}\,,
$$

where $\tilde{c}'_a = \tilde{c}_a + n_{\mathrm{cf}} \delta_{a,1}$ accounts for the shifted constant term.

In [ ]:
if HAS_CONI:
    # Conifold PFV: the pfv_to_moduli method handles the conifold case automatically
    M_coni = jnp.array([1., -3., 10.])  # Example M-vector
    K_coni = jnp.array([2., 1., -1.])   # Example K-vector
    
    try:
        flux_coni = model_coni.pfv_to_flux(M_coni, K_coni)
        print(f"Conifold PFV flux: {flux_coni.astype(int)}")
        
        tau_coni = 5.0j
        z_coni = model_coni.pfv_to_moduli(M_coni, K_coni, tau_coni)
        print(f"\nModuli at PFV locus:")
        print(f"  z_cf (conifold) = {z_coni[0]}")
        print(f"  z_bulk = {z_coni[1:]}")
        print(f"  |z_cf| = {float(jnp.abs(z_coni[0])):.4e}")
    except Exception as e:
        print(f"Conifold PFV construction: {e}")
        print("(This may require specific flux choices satisfying the PFV conditions.)")
else:
    print("Skipped (coniLCS model not available)")

### The ConifoldFreezer

The `ConifoldFreezer` integrates out the heavy conifold modulus $z_{\mathrm{cf}}$ by solving its equation of motion analytically. This reduces the effective field theory to the light bulk moduli $z^\alpha$ and $\tau$.

In [ ]:
if HAS_CONI:
    try:
        freezer = ConifoldFreezer(model_coni)
        print(f"ConifoldFreezer:")
        print(f"  Heavy indices: {freezer.heavy_indices}")
        print(f"  Light indices: {freezer.light_indices}")
        print(f"  n_heavy = {freezer.n_heavy}")
        print(f"  n_light = {freezer.n_light}")
    except Exception as e:
        print(f"ConifoldFreezer: {e}")
else:
    print("Skipped (coniLCS model not available)")

## Summary

| Approach | Mechanism | $|W_0|$ | Requirements |
|----------|-----------|---------|-------------|
| **Standard PFV** | $W_{\mathrm{poly}} = 0$ along $z^a = p^a\tau$ | $\sim 10^{-8}$ (racetrack) | PFV conditions on $(M^a, K_a)$ |
| **Conifold PFV** | Bulk PFV + logarithmic $z_{\mathrm{cf}}$ stabilisation | $\sim 10^{-8}$ + warped throat | PFV conditions on bulk + conifold charge |

**Key API methods:**
- `model.pfv_to_flux(M, K)` — construct full flux vector from PFV data
- `model.pfv_to_moduli(M, K, tau)` — compute moduli on the flat direction
- `model.flux_to_pfv(flux)` — extract $(M, K)$ from a flux vector
- `model.newton_method_flux_vacua(z, tau, flux, solver_mode="real")` — refine to exact vacuum
- `ConifoldFreezer(model)` — integrate out the conifold modulus

**Tips:**
- Use `solver_mode="real"` and `step_size_Newton=0.1` for PFV Newton refinement (the flat direction makes the full Newton step unstable)
- The gauge-invariant $|W_0|$ (`superpotential_gauge_invariant`) is the physically meaningful quantity
- Verify all PFV conditions numerically before running Newton